In [1]:
# Daftar kompetisi yang terpasang di Kaggle.
import os

for name in os.listdir('/kaggle/input/competitions'):
    print(name)

cassava-leaf-disease-classification


In [1]:
# Pastikan GPU tersedia sebelum training.
!nvidia-smi

Sun Nov  1 04:19:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
# GANTI DI SINI: pilih satu sesuai tugasmu
MODEL_CHOICE = "efficientnetb0"  # Adam: "mobilenetv2", Arif: "resnet50", Daffa: "efficientnetb0"
TRAINING_SEED = 42
NAMA_ANGGOTA = "Daffa"  # GANTI: Adam / Arif / Daffa

import platform, sys
import tensorflow as tf

print("Anggota:", NAMA_ANGGOTA)
print("Model:", MODEL_CHOICE)
print("Seed uji:", TRAINING_SEED)
print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

Anggota: Daffa
Model: efficientnetb0
Seed uji: 42
Python: 3.13.15
TensorFlow: 2.20.0
Keras: 3.13.2
GPU tersedia: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
# Verifikasi ulang manifest agar run memakai split yang sama.
from pathlib import Path
import hashlib
import pandas as pd

DATA_ROOT = Path("/kaggle/input/competitions/cassava-leaf-disease-classification")

# GANTI DI SINI dengan lokasi manifest di akunmu
MANIFEST = Path("/kaggle/input/datasets/wandarix/daun-singkong/Dataset/manifest_split_seed42.csv")
EXPECTED_SHA256 = "aac30db60c0f6172776a39d2077a9dba4e1377f6036c5bd8f510aee9b96f4b48"

print("DATA_ROOT ada:", DATA_ROOT.exists())
print("MANIFEST ada:", MANIFEST.exists())
actual_hash = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
print("SHA-256:", actual_hash)
assert actual_hash == EXPECTED_SHA256, "Manifest berbeda. Minta salinan master dari Arif."

manifest = pd.read_csv(MANIFEST)
print("Baris manifest:", len(manifest))
print(manifest["split"].value_counts())

DATA_ROOT ada: True
MANIFEST ada: True
SHA-256: aac30db60c0f6172776a39d2077a9dba4e1377f6036c5bd8f510aee9b96f4b48
Baris manifest: 21397
split
train         14978
validation     3210
test           3209
Name: count, dtype: int64


In [3]:
# Ukuran input dan batch disamakan untuk tiga model.
IMG_SIZE = 224
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS_SMOKE = 1
NUM_CLASSES = 5
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]

print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("LEARNING_RATE:", LEARNING_RATE)
print("EPOCHS_SMOKE:", EPOCHS_SMOKE)
print("Kelas:", CLASS_NAMES)

IMG_SIZE: 224
BATCH_SIZE: 32
LEARNING_RATE: 0.001
EPOCHS_SMOKE: 1
Kelas: ['CBB', 'CBSD', 'CGM', 'CMD', 'Healthy']


In [4]:
# Samakan label manifest dengan train.csv lalu petakan path gambar.
import numpy as np

train_csv = pd.read_csv(DATA_ROOT / "train.csv")
check = manifest.merge(train_csv, on="image_id", suffixes=("_m", "_c"), how="inner")
assert len(check) == len(manifest), "Ada image_id manifest yang tidak ada di train.csv."
assert (check["label_m"] == check["label_c"]).all(), "Ada label yang beda dengan train.csv."

train_df = manifest[manifest["split"] == "train"].copy()
val_df = manifest[manifest["split"] == "validation"].copy()
print("Train:", len(train_df), "| Validation:", len(val_df))
print(train_df["label"].value_counts().sort_index())
print(val_df["label"].value_counts().sort_index())

IMAGE_DIR = DATA_ROOT / "train_images"
train_paths = [(str(IMAGE_DIR / i), int(l)) for i, l in zip(train_df["image_id"], train_df["label"])]
val_paths = [(str(IMAGE_DIR / i), int(l)) for i, l in zip(val_df["image_id"], val_df["label"])]
print("Contoh train:", train_paths[0])
print("Contoh validation:", val_paths[0])

Train: 14978 | Validation: 3210
label
0     761
1    1532
2    1670
3    9211
4    1804
Name: count, dtype: int64
label
0     163
1     328
2     358
3    1974
4     387
Name: count, dtype: int64
Contoh train: ('/kaggle/input/competitions/cassava-leaf-disease-classification/train_images/1000201771.jpg', 3)
Contoh validation: ('/kaggle/input/competitions/cassava-leaf-disease-classification/train_images/1000015157.jpg', 0)


In [5]:
# Bobot dari train saja supaya info validation tidak bocor.
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight

train_labels = train_df["label"].to_numpy()
classes = np.array([0, 1, 2, 3, 4])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_labels)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print("Jumlah per kelas train:", dict(Counter(train_labels)))
print("Class weight:", class_weight)

Jumlah per kelas train: {np.int64(3): 9211, np.int64(1): 1532, np.int64(2): 1670, np.int64(0): 761, np.int64(4): 1804}
Class weight: {0: 3.9363994743758215, 1: 1.9553524804177547, 2: 1.7937724550898204, 3: 0.3252198458364998, 4: 1.6605321507760533}


In [6]:
# Preprocess mengikuti backbone; augmentasi hanya untuk train.
AUTOTUNE = tf.data.AUTOTUNE

if MODEL_CHOICE == "mobilenetv2":
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
elif MODEL_CHOICE == "resnet50":
    from tensorflow.keras.applications.resnet50 import preprocess_input
elif MODEL_CHOICE == "efficientnetb0":
    from tensorflow.keras.applications.efficientnet import preprocess_input
else:
    raise ValueError("MODEL_CHOICE harus mobilenetv2 / resnet50 / efficientnetb0")

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="augmentasi_train")

def load_and_preprocess(path, label, training=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    if training:
        img = augment(img)
    img = preprocess_input(img)
    return img, label

def make_ds(pairs, training, smoke_n=None):
    if smoke_n is not None:
        pairs = pairs[:smoke_n]
    paths, labels = zip(*pairs)
    ds = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    if training:
        ds = ds.shuffle(1000, seed=TRAINING_SEED)
    ds = ds.map(lambda p, l: load_and_preprocess(p, l, training=training), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

# Uji pipeline pakai sebagian kecil dulu supaya cepat
train_smoke = make_ds(train_paths, training=True, smoke_n=640)
val_smoke = make_ds(val_paths, training=False, smoke_n=320)
print("Batch train:", len(train_smoke), "| Batch validation:", len(val_smoke))
for x, y in train_smoke.take(1):
    print("Bentuk gambar satu batch:", x.shape)
    print("Bentuk label satu batch:", y.shape)
    print("Contoh label:", y.numpy()[:8])

I0000 00:00:1793512491.788285      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793512491.791702      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Batch train: 20 | Batch validation: 10
Bentuk gambar satu batch: (32, 224, 224, 3)
Bentuk label satu batch: (32,)
Contoh label: [3 3 3 3 2 3 3 3]


In [7]:
tf.keras.utils.set_random_seed(TRAINING_SEED)

if MODEL_CHOICE == "mobilenetv2":
    from tensorflow.keras.applications import MobileNetV2
    base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
elif MODEL_CHOICE == "resnet50":
    from tensorflow.keras.applications import ResNet50
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
elif MODEL_CHOICE == "efficientnetb0":
    from tensorflow.keras.applications import EfficientNetB0
    base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

base.trainable = False  # tahap uji: backbone dibekukan, hanya kepala klasifikasi yang belajar

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()
print("Output model harus 5 angka probabilitas per foto.")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,055,976 (15.47 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

Output model harus 5 angka probabilitas per foto.


In [8]:
# Uji satu epoch dan simpan checkpoint dengan val_loss terkecil.
import time

ckpt_path = "/kaggle/working/smoke_model.keras"
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, verbose=1),
]

t0 = time.time()
history = model.fit(
    train_smoke,
    validation_data=val_smoke,
    epochs=EPOCHS_SMOKE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)
print("Waktu:", round(time.time() - t0, 1), "detik")
print("Loss train:", history.history["loss"])
print("Loss validation:", history.history["val_loss"])

p = Path(ckpt_path)
print("Checkpoint ada:", p.exists(), "| Ukuran:", p.stat().st_size if p.exists() else 0, "bytes")
assert p.exists() and p.stat().st_size > 0, "Checkpoint tidak tersimpan."

2026-11-01 05:57:08.551141: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-01 05:57:08.695443: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-01 05:57:09.045251: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-01 05:57:09.188741: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-01 05:57:09.331252: E external/local_xla/xla/stream_

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.2543 - loss: 1.6761
Epoch 1: val_loss improved from None to 1.33341, saving model to /kaggle/working/smoke_model.keras

Epoch 1: finished saving model to /kaggle/working/smoke_model.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 38s 551ms/step - accuracy: 0.3391 - loss: 1.5646 - val_accuracy: 0.4500 - val_loss: 1.3334
Waktu: 37.8 detik
Loss train: [1.5645530223846436]
Loss validation: [1.3334076404571533]
Checkpoint ada: True | Ukuran: 17121354 bytes


In [9]:
# Pastikan output 5 probabilitas yang jumlahnya sekitar 1.
import numpy as np

for x, y in val_smoke.take(1):
    pred = model.predict(x)
    print("Bentuk prediksi:", pred.shape)
    print("Contoh probabilitas:", pred[0])
    print("Jumlah probabilitas satu foto:", pred[0].sum())
    print("Prediksi kelas:", np.argmax(pred[:5], axis=1))
    print("Label asli:", y.numpy()[:5])
    break
assert pred.shape[1] == 5, "Output harus 5 kelas."

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
Bentuk prediksi: (32, 5)
Contoh probabilitas: [0.23661539 0.2641126  0.13193043 0.13270767 0.23463397]
Jumlah probabilitas satu foto: 1.0
Prediksi kelas: [1 0 3 0 0]
Label asli: [0 0 3 2 3]
